In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [2]:
import os
import numpy as np
import pandas as pd

from src.anonymization import mdav_c, rmdav_m

In [3]:
# Input file
INPUT_PATH = "../data/case_embeddings.parquet"

# Output directory
OUTPUT_DIR = "../data/anonymized_outputs"

# Cluster sizes
K_VALUES = [5, 10, 25, 50, 100]

os.makedirs(OUTPUT_DIR, exist_ok=True)

In [4]:
df = pd.read_parquet(INPUT_PATH)
df = df[:1000]
print(df.shape)
df.head()

(1000, 768)


,0,1,2,3,4,5,6,7,8,9,...,758,759,760,761,762,763,764,765,766,767
case_id,,,,,,,,,,,,,,,,,,,,,
PMC3738355_01,-0.015325,-0.009899,-0.036830,-0.037385,0.093386,0.005113,-0.046486,0.020052,0.008637,0.048950,...,-0.022806,-0.023765,0.031298,0.068690,0.017421,-0.022707,-0.007395,-0.009242,0.055752,-0.008441
PMC5015624_01,0.003192,-0.050309,-0.051869,-0.001368,0.017927,0.011034,-0.033032,-0.026722,0.047942,-0.003823,...,-0.023348,0.030943,0.015994,0.053487,0.007882,-0.002215,-0.045054,0.082847,0.029237,-0.059586
PMC6381877_01,0.001285,-0.003055,-0.048088,0.033391,0.051446,0.028892,-0.032158,-0.041325,0.027464,0.016388,...,0.004046,0.044057,-0.011429,0.072100,0.060097,-0.023966,0.007983,0.014736,0.076301,-0.011417
PMC5912312_01,0.018228,-0.041158,-0.023238,0.033113,0.034552,0.037681,-0.011011,0.036308,0.031218,0.016701,...,-0.021764,0.026760,0.031395,0.036984,-0.022724,0.006445,0.015169,-0.043845,0.063211,-0.021389
PMC5912312_02,0.008789,-0.029656,-0.044781,0.039170,0.069089,0.047241,-0.010716,-0.030129,0.010307,0.017928,...,-0.005452,0.019038,-0.007263,0.078163,-0.018104,0.001024,0.010223,0.006496,0.016193,-0.043640


In [5]:
embedding_columns = sorted(
    [c for c in df.columns if str(c).isdigit()],
    key=int,
)

X = df[embedding_columns].to_numpy(dtype=float)

print(X.shape)

(1000, 768)


In [6]:
for k in K_VALUES:

    print(f"Running MDAV-C (k={k})...")

    X_mdav = mdav_c(X, k)

    output = df.copy()
    output[embedding_columns] = X_mdav

    output.to_parquet(
        f"{OUTPUT_DIR}/mdav_c_k{k}.parquet",
        index=False,
    )

Running MDAV-C (k=5)...
Running MDAV-C (k=10)...
Running MDAV-C (k=25)...
Running MDAV-C (k=50)...
Running MDAV-C (k=100)...


In [7]:
for k in K_VALUES:

    print(f"Running RMDAV-M (k={k})...")

    X_rmdav = rmdav_m(
        X,
        k,
        random_state=42,
    )

    output = df.copy()
    output[embedding_columns] = X_rmdav

    output.to_parquet(
        f"{OUTPUT_DIR}/rmdav_m_k{k}.parquet",
        index=False,
    )

Running RMDAV-M (k=5)...
Running RMDAV-M (k=10)...
Running RMDAV-M (k=25)...
Running RMDAV-M (k=50)...
Running RMDAV-M (k=100)...


In [8]:
print("Finished.")

print(f"Saved outputs to: {OUTPUT_DIR}")

Finished.
Saved outputs to: ../data/anonymized_outputs
